## SQG fields visualization

Loads a trajectory from `datasets/SQG/full_path/path_*.pt` (generated by `scripts/SQG/generate_fields.py`) and visualizes frames, time series, and an animation.


In [ ]:
import os
import torch
import matplotlib.pyplot as plt

from project_utils.paths import dataset_trajectory_dir

# Must match data_root / trajectory_subdir in your YAML (generator + dataset_kwargs).
DATA_ROOT = 'SQG'
TRAJECTORY_SUBDIR = 'full_path'
path = os.path.join(dataset_trajectory_dir(DATA_ROOT, TRAJECTORY_SUBDIR), 'path_0.pt')
print('Loading:', path)
fields = torch.load(path, map_location='cpu')
print('shape:', tuple(fields.shape), 'dtype:', fields.dtype, 'min/max:', fields.min().item(), fields.max().item())


In [ ]:
# Pick a time index and plot both boundary channels
t = 0  # change me
fig, axs = plt.subplots(1, 2, figsize=(10, 4))
for c in range(2):
    im = axs[c].imshow(fields[t, c].numpy(), cmap='RdBu_r')
    axs[c].set_title(f't={t}, channel={c}')
    axs[c].axis('off')
    fig.colorbar(im, ax=axs[c], fraction=0.046, pad=0.04)
plt.tight_layout()
plt.show()


In [ ]:
# Plot a small temporal sequence for one channel at a fixed pixel location
c = 0
i = fields.shape[-2] // 2
j = fields.shape[-1] // 2
series = fields[:, c, i, j].numpy()
plt.figure(figsize=(8, 3))
plt.plot(series)
plt.title(f'Pixel time series (channel={c}, i={i}, j={j})')
plt.xlabel('t index')
plt.ylabel('value')
plt.grid(True)
plt.show()


In [ ]:
# Animate the SQG path (both boundary channels)
from matplotlib.animation import FuncAnimation
from IPython.display import HTML

T = fields.shape[0]
interval_ms = 200  # pause between frames (milliseconds)
max_frames = 200  # use at most this many consecutive frames (0, 1, 2, ...)
n_frames = min(T, max_frames)
frame_ids = list(range(n_frames))

vmin = float(fields.min())
vmax = float(fields.max())

fig, axs = plt.subplots(1, 2, figsize=(8, 4))
ims = []
cbars = []
for c in range(2):
    ims.append(
        axs[c].imshow(
            fields[0, c].numpy(),
            cmap='RdBu_r',
            vmin=vmin,
            vmax=vmax,
            animated=True,
        )
    )
    axs[c].set_title(f'channel {c}')
    axs[c].axis('off')
    cbars.append(fig.colorbar(ims[c], ax=axs[c], fraction=0.046, pad=0.04))
    cbars[c].set_label('PV')

title = fig.suptitle(f't = 0 / {n_frames - 1}  (showing {n_frames} of {T} frames)')

def update(frame_idx):
    t = frame_ids[frame_idx]
    for c in range(2):
        ims[c].set_array(fields[t, c].numpy())
    title.set_text(
        f't = {t}  (consecutive {frame_idx + 1}/{n_frames}; trajectory has {T} frames)'
    )
    return [*ims, title]

# blit=False keeps colorbars stable in the saved HTML animation
anim = FuncAnimation(
    fig,
    update,
    frames=len(frame_ids),
    interval=interval_ms,
    blit=False,
    repeat=True,
)
plt.close(fig)
HTML(anim.to_jshtml())
